# Export YOLOv8n to ONNX for Triton

Run this notebook inside the Triton Control code-server workspace. It downloads YOLOv8n, exports it to ONNX in a temporary directory, and places only the Triton artifact in `yolov8_onnx/1/model.onnx`.

## Install export dependencies

In [ ]:
%pip install ultralytics onnx onnxsim

## Export YOLOv8n

In [ ]:
import os
from pathlib import Path
import shutil
import tempfile

from ultralytics import YOLO

target = Path("yolov8_onnx/1/model.onnx")
target.parent.mkdir(parents=True, exist_ok=True)
target.unlink(missing_ok=True)

original_cwd = Path.cwd()
with tempfile.TemporaryDirectory() as export_dir:
    os.chdir(export_dir)
    try:
        model = YOLO("yolov8n.pt")
        export_path = model.export(
            format="onnx",
            imgsz=640,
            opset=12,
            simplify=True,
            dynamic=False,
        )
        exported_model = Path(export_path).resolve()
    finally:
        os.chdir(original_cwd)

    shutil.move(str(exported_model), target)

target, target.exists(), target.stat().st_size

## Inspect ONNX inputs and outputs

In [ ]:
import onnx

onnx_model = onnx.load("yolov8_onnx/1/model.onnx")

print("Inputs:")
for value in onnx_model.graph.input:
    print("-", value.name)

print("Outputs:")
for value in onnx_model.graph.output:
    print("-", value.name)